# MobileNet-V3-Large + EfficientNet-B0 Ensemble — Iteration 02

## Objective
Ensemble the best MobileNet-V3-Large checkpoint (Iter 01, Val AUC 0.9188, Test F2 0.6492) with the best EfficientNet-B0 checkpoint (Iter 06, Val AUC 0.9167, Test F2 0.6952) by averaging their TTA probabilities before thresholding. No additional training is performed — both models are loaded from their saved checkpoints. The hypothesis is that combining two independently-trained architectures with different inductive biases (depthwise separable convolutions vs. compound-scaled MBConv) will produce a more calibrated probability estimate than either model alone, pushing Test F2 above the EfficientNet-B0 single-model baseline of 0.6952.

## Architecture

| Component | EfficientNet-B0 Iter 06 | MobileNetV3-Large Iter 01 | Ensemble (this notebook) |
|---|---|---|---|
| Model | `EfficientNetB0WithMetadata` | `MobileNetV3LargeWithMetadata` | Both (averaged) |
| Image features | 1280-dim | 960-dim | — |
| Patient metadata | dim=17 | dim=17 | dim=17 |
| Unfrozen | Last 6 blocks | Full backbone | — (inference only) |
| Trainable params | ~4M | ~3M | — |
| Dropout | 0.5 | 0.5 | — |
| L1 lambda | 1e-3 | 1e-3 | — |
| TTA at inference | 8× | 8× | 8× per model, then average |
| Ensemble strategy | — | — | **Mean of per-model TTA probabilities** |
| Val AUC (checkpoint) | 0.9167 | 0.9188 | — |
| Test F2 (individual) | 0.6952 | 0.6492 | TBD |

## Hypothesis
EfficientNet-B0 has a higher Test F2 (0.6952) but MobileNetV3-Large has a higher saved Val AUC (0.9188). These two architectures make errors on different samples due to their different feature extraction strategies. Averaging their softmax probabilities after 8× TTA should smooth out individual model uncertainty and produce more robust calibration, particularly for borderline cases near the decision boundary. The ensemble is expected to achieve Test F2 ≥ 0.70.

## Import libraries, set seed, and choose device

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from sklearn.metrics import fbeta_score

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'src').exists())
sys.path.insert(0, str(ROOT))

from src.data.dataset import HAM10000DatasetWithMetadata
from src.models.efficientnet import EfficientNetB0WithMetadata
from src.models.mobilenet import MobileNetV3LargeWithMetadata
from src.utils import seed_everything

import pandas as pd

g = seed_everything(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

## Load and split data

In [ ]:
METADATA_PATH      = str(ROOT / 'data_new/raw/HAM10000_metadata')
TEST_METADATA_PATH = str(ROOT / 'data_new/raw/ISIC2018_Task3_Test_GroundTruth.csv')

val_dataset_raw = HAM10000DatasetWithMetadata(
    csv_path=str(ROOT / 'data_new/splits/val.csv'),
    image_dir=str(ROOT / 'data_new/images/train'),
    metadata_path=METADATA_PATH,
    transform=None,
)

test_dataset_raw = HAM10000DatasetWithMetadata(
    csv_path=str(ROOT / 'data_new/splits/test.csv'),
    image_dir=str(ROOT / 'data_new/images/test'),
    metadata_path=TEST_METADATA_PATH,
    transform=None,
)

print(f'Val: {len(val_dataset_raw)} | Test: {len(test_dataset_raw)}')

## Model Definition

In [ ]:
METADATA_DIM = 17

# EfficientNet-B0 — last 6 blocks unfrozen, dropout=0.5
efficientnet = EfficientNetB0WithMetadata(
    metadata_dim=METADATA_DIM,
    num_classes=1,
    freeze_backbone=True,
    dropout=0.5,
).to(device)
efficientnet.load_state_dict(
    torch.load(str(ROOT / 'models/efficientnet_b0_l1_metadata_best.pth'), map_location=device)
)
efficientnet.eval()
print('EfficientNet-B0 loaded.')

# MobileNetV3-Large — full backbone unfrozen, dropout=0.5
mobilenet = MobileNetV3LargeWithMetadata(
    metadata_dim=METADATA_DIM,
    num_classes=1,
    freeze_backbone=False,
    dropout=0.5,
).to(device)
mobilenet.load_state_dict(
    torch.load(str(ROOT / 'models/01.mobilenet_v3_metadata_best.pth'), map_location=device)
)
mobilenet.eval()
print('MobileNetV3-Large loaded.')

## Threshold Tuning (Best Val F2)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _base(extra=None):
    ops = [transforms.Resize((224, 224))]
    if extra:
        ops += extra
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)

tta_transforms = [
    _base(),                                                        # 1. identity
    _base([transforms.RandomHorizontalFlip(p=1.0)]),                # 2. H-flip
    _base([transforms.RandomVerticalFlip(p=1.0)]),                  # 3. V-flip
    _base([transforms.RandomHorizontalFlip(p=1.0),
           transforms.RandomVerticalFlip(p=1.0)]),                  # 4. HV-flip
    _base([transforms.RandomRotation(degrees=(90, 90))]),           # 5. rotate 90
    _base([transforms.RandomRotation(degrees=(180, 180))]),         # 6. rotate 180
    _base([transforms.RandomRotation(degrees=(270, 270))]),         # 7. rotate 270
    _base([transforms.RandomRotation(degrees=(45, 45))]),           # 8. rotate 45
]


def tta_predict_single(model, dataset, device, tta_transforms):
    """Run TTA for a single metadata-fusion model. Returns (probs, labels)."""
    all_probs, all_labels = [], []
    with torch.no_grad():
        for idx in range(len(dataset)):
            pil_img, metadata, label = dataset[idx]
            metadata = metadata.unsqueeze(0).to(device)
            preds = [
                torch.sigmoid(model(t(pil_img).unsqueeze(0).to(device), metadata)).item()
                for t in tta_transforms
            ]
            all_probs.append(np.mean(preds))
            all_labels.append(label)
    return np.array(all_probs), np.array(all_labels)


print('Running TTA on validation set (EfficientNet-B0)...')
val_probs_eff, val_labels = tta_predict_single(efficientnet, val_dataset_raw, device, tta_transforms)

print('Running TTA on validation set (MobileNetV3-Large)...')
val_probs_mob, _          = tta_predict_single(mobilenet,    val_dataset_raw, device, tta_transforms)

val_probs_ensemble = (val_probs_eff + val_probs_mob) / 2.0

thresholds = np.arange(0.01, 0.90, 0.01)
f2_scores  = [
    fbeta_score(val_labels, (val_probs_ensemble >= t).astype(int), beta=2, pos_label=1, zero_division=0)
    for t in thresholds
]
best_threshold = thresholds[np.argmax(f2_scores)]
print(f'Best threshold: {best_threshold:.2f} | Val F2: {max(f2_scores):.4f}')

## Test Set Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, classification_report

print('Running TTA on test set (EfficientNet-B0)...')
test_probs_eff, test_labels = tta_predict_single(efficientnet, test_dataset_raw, device, tta_transforms)

print('Running TTA on test set (MobileNetV3-Large)...')
test_probs_mob, _           = tta_predict_single(mobilenet,    test_dataset_raw, device, tta_transforms)

test_probs_ensemble = (test_probs_eff + test_probs_mob) / 2.0
all_preds = (test_probs_ensemble >= best_threshold).astype(int)

auc     = roc_auc_score(test_labels, test_probs_ensemble)
bal_acc = balanced_accuracy_score(test_labels, all_preds)
f2      = fbeta_score(test_labels, all_preds, beta=2, pos_label=1, zero_division=0)

print(f'Threshold:          {best_threshold:.2f}')
print(f'AUC-ROC:            {auc:.4f}')
print(f'Balanced Accuracy:  {bal_acc:.4f}')
print(f'F2 Score:           {f2:.4f}')
print()
print(classification_report(test_labels, all_preds, target_names=['Non-Melanoma', 'Melanoma'], digits=4))